# 🚐 CHIVA TOURS - PIPELINE AUTOMATIZADO
## Scraping + Limpieza + Envío de Emails

---

**Objetivo:** Automatizar contacto a hoteles/Airbnbs de Medellín para ofrecer servicio de chiva tours.

**Modelo:**
- Precio servicio: $70k COP
- Comisión hotel/Airbnb: $20k
- Tu ganancia: $50k


## PARTE 1: SETUP INICIAL

In [ ]:
# Instalar librerías necesarias
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("📦 Instalando librerías...")
install("pandas")
install("beautifulsoup4")
install("requests")
print("✅ Librerías instaladas")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import time
import random
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from datetime import datetime
import json
import os
from pathlib import Path

print("✅ Imports completados")

## PARTE 2: CARGAR DATOS

In [ ]:
def cargar_datos_simulados():
    """
    Para scraping REAL de Google Maps:
    - Necesitas Selenium + ChromeDriver
    - O Google Maps API
    
    Por ahora usamos datos simulados para testing.
    """
    
    print("🔍 Cargando datos de hoteles/Airbnbs (simulado)...\n")
    
    hoteles = [
        {
            'nombre': 'Hotel Dann Carlton',
            'email': 'reservas@danncarltonsabaneta.com',
            'telefono': '+573001234567',
            'tipo': 'hotel',
            'ubicacion': 'Sabaneta'
        },
        {
            'nombre': 'Airbnb Centro - Juan Carlos',
            'email': 'juan.carlos.mendez@gmail.com',
            'telefono': '+573105432109',
            'tipo': 'airbnb',
            'ubicacion': 'Centro'
        },
        {
            'nombre': 'Hotel Estelar Medellin',
            'email': 'info@hotelestelar.com',
            'telefono': '+573012345678',
            'tipo': 'hotel',
            'ubicacion': 'Laureles'
        },
        {
            'nombre': 'Airbnb Laureles - Maria',
            'email': 'maria.rodriguez@outlook.com',
            'telefono': '+573167890123',
            'tipo': 'airbnb',
            'ubicacion': 'Laureles'
        },
        {
            'nombre': 'Hotel Arví Park',
            'email': 'contacto@arvipark.com.co',
            'telefono': '+573019876543',
            'tipo': 'hotel',
            'ubicacion': 'Envigado'
        },
        {
            'nombre': 'Hostel Medellín Central',
            'email': 'info@hostelmedellín.com',
            'telefono': '+573001111111',
            'tipo': 'hostal',
            'ubicacion': 'Centro'
        }
    ]
    
    df = pd.DataFrame(hoteles)
    print(f"✅ {len(df)} hoteles/Airbnbs cargados\n")
    
    return df

# Ejecutar
df = cargar_datos_simulados()
print(df)

## PARTE 3: LIMPIAR DATOS

In [ ]:
def limpiar_datos(df):
    """
    - Elimina duplicados
    - Valida emails
    - Añade columnas de control
    """
    
    print("🧹 Limpiando datos...\n")
    
    # Eliminar duplicados
    df_inicial = len(df)
    df = df.drop_duplicates(subset=['email'])
    print(f"  - Duplicados eliminados: {df_inicial - len(df)}")
    
    # Validar emails
    df['email_valido'] = df['email'].str.contains('@', regex=False)
    emails_invalidos = len(df[df['email_valido'] == False])
    print(f"  - Emails inválidos eliminados: {emails_invalidos}")
    df = df[df['email_valido'] == True].drop('email_valido', axis=1)
    
    # Agregar columnas de control
    df['estado'] = 'sin_contactar'  # sin_contactar, enviado, interesado, cliente
    df['fecha_intento'] = None
    df['fecha_respuesta'] = None
    df['respuesta'] = None
    df['referrals_completados'] = 0
    df['ganado'] = 0  # $20k × referrals
    
    print(f"\n✅ Datos limpios: {len(df)} contactos válidos\n")
    
    return df

# Ejecutar
df = limpiar_datos(df)
print("\n📋 DATOS FINALES:")
print(df[['nombre', 'tipo', 'email', 'estado']].to_string())

## PARTE 4: CREAR TEMPLATES DE EMAIL

In [ ]:
def crear_email_html(nombre_hotel):
    """
    Crea email HTML personalizado y profesional
    """
    
    html = f"""
    <!DOCTYPE html>
    <html>
        <head>
            <meta charset="UTF-8">
            <style>
                body {{
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                    line-height: 1.6;
                    color: #333;
                }}
                .container {{
                    max-width: 600px;
                    margin: 0 auto;
                }}
                .header {{
                    background: linear-gradient(135deg, #d4a574 0%, #c1956b 100%);
                    padding: 30px 20px;
                    color: white;
                    text-align: center;
                    border-radius: 8px 8px 0 0;
                }}
                .header h1 {{
                    margin: 0;
                    font-size: 28px;
                }}
                .content {{
                    padding: 30px 20px;
                    background: #f9f9f9;
                }}
                .offer {{
                    background: white;
                    padding: 20px;
                    margin: 20px 0;
                    border-left: 4px solid #d4a574;
                    border-radius: 4px;
                }}
                .offer h3 {{
                    color: #d4a574;
                    margin-top: 0;
                }}
                .offer ul {{
                    margin: 10px 0;
                    padding-left: 20px;
                }}
                .gain {{
                    background: #e8f5e9;
                    padding: 20px;
                    margin: 20px 0;
                    border-left: 4px solid #4caf50;
                    border-radius: 4px;
                    text-align: center;
                }}
                .gain h2 {{
                    color: #4caf50;
                    margin: 0;
                    font-size: 24px;
                }}
                .cta {{
                    display: inline-block;
                    background: #d4a574;
                    color: white;
                    padding: 12px 30px;
                    text-decoration: none;
                    border-radius: 5px;
                    margin: 10px 5px;
                }}
                .contact {{
                    text-align: center;
                    margin: 20px 0;
                }}
                .footer {{
                    color: #999;
                    font-size: 12px;
                    text-align: center;
                    padding: 20px;
                    border-top: 1px solid #eee;
                    background: #f9f9f9;
                    border-radius: 0 0 8px 8px;
                }}
            </style>
        </head>
        <body>
            <div class="container">
                <div class="header">
                    <h1>🚐 Chiva Tours Medellín</h1>
                    <p>Experiencias auténticas para tus huéspedes</p>
                </div>
                
                <div class="content">
                    <p>Hola <strong>{nombre_hotel}</strong>,</p>
                    
                    <p>Sabemos que tus huéspedes buscan experiencias auténticas y únicas en Medellín. Nosotros ofrecemos exactamente eso.</p>
                    
                    <div class="offer">
                        <h3>🚐 Experiencia Chiva Tour</h3>
                        <ul>
                            <li><strong>Precio:</strong> $70.000 COP por persona</li>
                            <li><strong>Duración:</strong> 4 horas</li>
                            <li><strong>Incluye:</strong> Guía local experto + ruta complete + fotos</li>
                            <li><strong>Punto de partida:</strong> Flexible (te coordinamos)</li>
                        </ul>
                    </div>
                    
                    <div class="gain">
                        <h2>💰 TÚ GANAS $20.000 COP</h2>
                        <p>Por cada reserva que hagas desde tu hotel/Airbnb</p>
                        <p style="font-size: 14px; margin: 10px 0 0 0;"><strong>Cero inversión. Cero compromiso. Solo comisión.</strong></p>
                    </div>
                    
                    <p><strong>¿Cómo funciona?</strong></p>
                    <ol>
                        <li>Un huésped te pregunta: "¿Dónde puedo hacer tours en Medellín?"</li>
                        <li>Tú lo referes con nosotros (un simple WhatsApp)</li>
                        <li>Nosotros coordinamos todo y cobramos $70k</li>
                        <li>Tú recibes $20k automáticamente</li>
                        <li>¡Repeat infinitamente!</li>
                    </ol>
                    
                    <div class="contact">
                        <p><strong>Contáctanos:</strong></p>
                        <p>📱 <strong>WhatsApp:</strong> <a href="https://wa.me/573001234567" class="cta">Escríbenos aquí</a></p>
                        <p>📧 <strong>Email:</strong> tuchiva@gmail.com</p>
                        <p>☎️ <strong>Tel:</strong> +57 300 123 4567</p>
                    </div>
                </div>
                
                <div class="footer">
                    <p>Este es una oportunidad de partnership sin compromiso. Si no está interesado, simplemente ignore este mensaje.</p>
                    <p>© 2024 Chiva Tours Medellín | Todos los derechos reservados</p>
                </div>
            </div>
        </body>
    </html>
    """
    
    return html

# Test: mostrar uno
print("✅ Template creado")
print("\nPrimeras líneas del email:")
print(crear_email_html("Hotel Test")[:500] + "...")

## PARTE 5: ENVÍO DE EMAILS (SIMULADO PARA TESTING)

In [ ]:
def enviar_emails_test(df, limite=3):
    """
    MODO TEST: Simula el envío sin realmente enviar
    
    Cuando estés listo para PRODUCCIÓN, descomenta la sección de envío real.
    """
    
    print(f"📧 Preparando envío a {min(limite, len(df))} contactos...\n")
    print("="*70)
    
    contactos_sin_enviar = df[df['estado'] == 'sin_contactar'].head(limite)
    
    for idx, contacto in contactos_sin_enviar.iterrows():
        print(f"\n📨 EMAIL #{idx+1}")
        print(f"   TO: {contacto['email']}")
        print(f"   Hotel/Airbnb: {contacto['nombre']}")
        print(f"   Tipo: {contacto['tipo'].upper()}")
        print(f"   Ubicación: {contacto['ubicacion']}")
        
        asunto = f"🚐 Rentabilidad para {contacto['nombre']} - Chiva Tours"
        print(f"   Asunto: {asunto}")
        
        # Crear mensaje
        msg = MIMEMultipart('alternative')
        msg['Subject'] = asunto
        msg['From'] = 'tuchiva@gmail.com'
        msg['To'] = contacto['email']
        
        # Cuerpo HTML
        html = crear_email_html(contacto['nombre'])
        msg.attach(MIMEText(html, 'html'))
        
        print(f"   ✅ Email creado (SIMULADO - no enviado)")
        
        # DESCOMENTA PARA ENVÍO REAL:
        # try:
        #     server = smtplib.SMTP('smtp.gmail.com', 587)
        #     server.starttls()
        #     server.login('tuchiva@gmail.com', 'tu_app_password')
        #     server.sendmail(msg['From'], msg['To'], msg.as_string())
        #     server.quit()
        #     print(f"   ✅ Email enviado correctamente")
        #     df.at[idx, 'estado'] = 'enviado'
        #     df.at[idx, 'fecha_intento'] = datetime.now().strftime('%Y-%m-%d %H:%M')
        # except Exception as e:
        #     print(f"   ❌ Error: {e}")
        
        time.sleep(2)  # Simular espera entre emails
    
    print("\n" + "="*70)
    print("\n✅ Simulación de envío completada\n")
    
    return df

# Ejecutar
df = enviar_emails_test(df, limite=2)

## PARTE 6: DASHBOARD Y MÉTRICAS

In [ ]:
def mostrar_dashboard(df):
    """
    Muestra métricas clave del pipeline
    """
    
    print("\n" + "="*70)
    print("📊 DASHBOARD CHIVA TOURS")
    print("="*70 + "\n")
    
    # Métricas
    total = len(df)
    sin_contactar = len(df[df['estado'] == 'sin_contactar'])
    enviados = len(df[df['estado'] == 'enviado'])
    interesados = len(df[df['estado'] == 'interesado'])
    clientes = len(df[df['estado'] == 'cliente'])
    
    print(f"📋 CONTACTOS:")
    print(f"   Total: {total}")
    print(f"   ⚪ Sin contactar: {sin_contactar}")
    print(f"   📤 Enviados: {enviados}")
    print(f"   💬 Interesados: {interesados}")
    print(f"   ✅ Clientes activos: {clientes}")
    
    # Tasas
    if enviados > 0:
        tasa_respuesta = (interesados / enviados) * 100
        print(f"\n📈 TASAS:")
        print(f"   Tasa de respuesta: {tasa_respuesta:.1f}% ({interesados}/{enviados})")
    
    if clientes > 0:
        tasa_conversion = (clientes / enviados) * 100 if enviados > 0 else 0
        print(f"   Tasa de conversión: {tasa_conversion:.1f}% ({clientes}/{enviados})")
    
    # Ingresos
    if clientes > 0:
        referrals_total = df['referrals_completados'].sum()
        ganado_total = df['ganado'].sum()
        print(f"\n💰 INGRESOS:")
        print(f"   Referrals totales: {referrals_total}")
        print(f"   Ganado total: ${ganado_total:,} COP")
        print(f"   Promedio por cliente: ${ganado_total/clientes:,.0f} COP")
    
    # Top performers
    top_contactos = df[df['referrals_completados'] > 0].nlargest(3, 'ganado')
    if len(top_contactos) > 0:
        print(f"\n🏆 TOP CONTACTOS:")
        for i, (idx, row) in enumerate(top_contactos.iterrows(), 1):
            print(f"   {i}. {row['nombre']}: ${row['ganado']:,} ({row['referrals_completados']} referrals)")
    
    # Por tipo
    print(f"\n📊 POR TIPO:")
    tipos = df.groupby('tipo').size()
    for tipo, cantidad in tipos.items():
        print(f"   {tipo.capitalize()}: {cantidad}")
    
    print("\n" + "="*70 + "\n")
    
    return {
        'total': total,
        'sin_contactar': sin_contactar,
        'enviados': enviados,
        'interesados': interesados,
        'clientes': clientes
    }

# Ejecutar
metricas = mostrar_dashboard(df)

## PARTE 7: GUARDAR Y DESCARGAR

In [ ]:
# Guardar en CSV
archivo_csv = 'contactos_chiva_tours.csv'
df.to_csv(archivo_csv, index=False, encoding='utf-8')
print(f"✅ Datos guardados en: {archivo_csv}")
print(f"   Tamaño: {os.path.getsize(archivo_csv) / 1024:.1f} KB\n")

# Mostrar preview
print("📋 PREVIEW DEL ARCHIVO:")
print(df[['nombre', 'tipo', 'email', 'estado', 'referrals_completados', 'ganado']].to_string())

In [ ]:
# Para descargar en Colab
try:
    from google.colab import files
    print("\n⬇️ Descargando archivo desde Colab...\n")
    files.download(archivo_csv)
    print("✅ Descarga completada\n")
except:
    print("(No estás en Colab o ya descargó)")

## PARTE 8: PRÓXIMOS PASOS

### ✅ LO QUE ACABAS DE HACER:

1. **Cargar datos** de hoteles/Airbnbs
2. **Limpiar** emails y duplicados
3. **Crear templates** HTML personalizados
4. **Simular envío** de emails
5. **Mostrar dashboard** con métricas
6. **Guardar CSV** para descargar

---

### 🚀 PARA PRODUCCIÓN:

#### 1. **Obtener Gmail App Password:**
- Ve a: https://myaccount.google.com/apppasswords
- Selecciona: Mail + Windows/Mac/Linux
- Copia el password de 16 caracteres

#### 2. **Descomenta el código de envío real:**
- En PARTE 5, descomenta la sección con `server.sendmail()`
- Reemplaza:
  - `tuchiva@gmail.com` → Tu email
  - `tu_app_password` → El password de 16 caracteres

#### 3. **Reemplaza datos simulados con scraping real:**
- Google Maps API o Selenium
- Booking.com scraping
- Directorios (ACOTUR, Cámara de Comercio)

#### 4. **Sube a GitHub:**
```bash
git init
git add .
git commit -m "Initial chiva bot"
git push origin main
```

#### 5. **Configura GitHub Actions:**
- Crea `.github/workflows/chiva-daily.yml`
- Se ejecutará automáticamente cada día

---

### ⚠️ IMPORTANTE:

- **NUNCA** hardcodees credenciales en el código
- Usa variables de entorno o GitHub Secrets
- Respeta los ToS de cada plataforma
- No hagas spam (3-5 seg entre emails, máx 100/día)

---

### 📞 ¿PREGUNTAS?

Si necesitas ayuda con:
- Scraping real
- GitHub Actions
- Google Apps Script
- CRM en Sheets/Airtable

**Avísame y lo armamos juntos.**